# QC Filters

**Pinned Environment:** [`envs/sc-spatial.yaml`](../.../envs/sc-spatial.yaml)  

In [ ]:
import os
from pathlib import Path
import matplotlib.pyplot as plt
import anndata as ad
import scanpy as sc
import sys

### Set paths, import data

In [ ]:
sys.path.append(str(Path.cwd().resolve().parents[1]))

from config.paths import BASE_DIR
base_dir = BASE_DIR

input_dir = base_dir / "data/h5ad/export_01/01a_raw/output_folder"
output_dir = base_dir / "data/h5ad/export_01/01b_filter"

output_dir.mkdir(parents=True, exist_ok=True)

Make a list of files in the directory:

In [ ]:
sample_list = os.listdir(input_dir)
sample_list

Add samples to list if they end in .h5ad:

In [ ]:
sample_files = [
    os.path.join(input_dir, f)
    for f in os.listdir(input_dir)
    if f.endswith(".h5ad") and f.startswith("TMA")
]

adata_list = [ad.read_h5ad(f) for f in sample_files]

for f, adata in zip(sample_files, adata_list):
    print(f"{os.path.basename(f)} > {adata.n_obs:,} cells × {adata.n_vars:,} genes")

## Filter

Filtering based on total cell count thresholds from Max's paper.

In [ ]:
def plot_violin_subplots(adata_list, obs_keys):
    n = len(adata_list)
    fig, axs = plt.subplots(1, n, figsize=(6 * n, 5))

    if n == 1:
        axs = [axs]  # make iterable if only one sample

    for ax, adata in zip(axs, adata_list):
        sc.pl.violin(adata, obs_keys, show=False, ax=ax)
        ax.set_title(getattr(adata, "filename", "Sample"))

    plt.tight_layout()
    plt.show()

In [ ]:
plot_violin_subplots(adata_list, ["total_counts"])

In [ ]:
# Total counts – Floor threshold
for i, adata in enumerate(adata_list):
    sample_id = adata.obs["sample_id"].unique()[0]
    print(f"{sample_id} shape before total_counts floor threshold: {adata.shape}")
    adata_list[i] = adata[adata.obs["total_counts"] > 50].copy()
    print(
        f"{sample_id} shape after total_counts floor threshold: {adata_list[i].shape}"
    )

In [ ]:
# Total counts - ceiling threshold
for i, adata in enumerate(adata_list):
    sample_id = adata.obs["sample_id"].unique()[0]
    print(f"{sample_id} shape before total_counts ceiling threshold: {adata.shape}")
    total_counts_ceiling = adata.obs["total_counts"].quantile(0.9975)
    adata_list[i] = adata[adata.obs["total_counts"] < total_counts_ceiling, :].copy()
    print(
        f"{sample_id} shape after total_counts ceiling threshold: {adata_list[i].shape}"
    )

In [ ]:
plot_violin_subplots(adata_list, ["total_counts"])

## Concatenate

In [ ]:
adata = ad.concat(adata_list, join="outer", label="batch", index_unique="-")

In [ ]:
adata.obs.sample_id.value_counts()

## Normalization

In [ ]:
adata.layers["counts"] = adata.X.copy()

In [ ]:
def remove_transgenes(adata, pattern="Seq"):
    print(f"\nProcessing: {getattr(adata, 'uns', {}).get('name', 'AnnData')}")
    print(f"Initial # genes: {adata.n_vars}")

    # Identify transgenes (always exactly five)
    transgenes = adata.var_names[
        adata.var_names.str.contains(pattern, case=False, na=False)
    ]
    print(f"Transgenes: {list(transgenes)}")

    # Remove them
    adata_filtered = adata[:, ~adata.var_names.isin(transgenes)].copy()
    print(f"Remaining genes: {adata_filtered.n_vars}")

    return adata_filtered


adata_subset = remove_transgenes(adata, pattern="Seq")

In [ ]:
sc.pp.normalize_total(adata_subset)
adata_subset.layers["normalized"] = adata_subset.X.copy()

sc.pp.log1p(adata_subset)
adata_subset.raw = adata_subset.copy()
adata_subset.layers["log1p"] = adata_subset.X.copy()

In [ ]:
print(adata.layers)
print(adata_subset.layers)

## Export

In [ ]:
# 480 genes
filename = os.path.join(output_dir, "adata-480.h5ad")
os.makedirs(os.path.dirname(filename), exist_ok=True)
adata.write_h5ad(filename, compression="gzip")
print(f"Filtered AnnData with all genes saved to: {filename}")

# 475 genes
filename = os.path.join(output_dir, "adata-475.h5ad")
os.makedirs(os.path.dirname(filename), exist_ok=True)
adata_subset.write_h5ad(filename, compression="gzip")
print(f"Filtered AnnData without transgenes saved to: {filename}")